# 第 10 章 · Bash Environment、超时与进程组

**这一章你会得到什么**：把 Environment 这一层吃透——cwd、环境变量、返回码、超时，以及为什么超时要杀**整个进程组**。

## 📖 对照源码（在 IDE 里打开这些文件，边看边跑）

- `src/minisweagent/environments/local.py` **L24–43** — `execute()`（cwd/env/返回码/异常）
- `src/minisweagent/environments/local.py` **L45–56** — `_check_finished()`
- `src/minisweagent/environments/local.py` **L72–92** — `_run()`（subprocess + 超时 killpg 进程组）

> 快捷：代码格里 `函数名??` 直接打印源码；或用 `show_source("相对路径", 起始行, 结束行)`。

In [ ]:
import os, sys
from pathlib import Path
os.environ["MSWEA_SILENT_STARTUP"] = "1"
REPO = Path(r"/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent")
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
import minisweagent
print("mini-SWE-agent:", minisweagent.__version__)

In [ ]:
def show_source(rel_path: str, start: int, end: int) -> None:
    lines = (REPO / rel_path).read_text().splitlines()
    end = min(end, len(lines))
    w = len(str(end))
    for n in range(start, end + 1):
        print(f"{n:>{w}}  {lines[n - 1]}")

## 实验 1：返回码与输出

`execute` 返回 `{output, returncode, exception_info}`。跑一个失败命令看 returncode。

In [ ]:
from minisweagent.environments.local import LocalEnvironment
env = LocalEnvironment()
print("成功:", env.execute({"command": "echo hello"}))
print("失败:", env.execute({"command": "exit 3"}))

## 实验 2：注入环境变量与工作目录

配置里的 `env` 和 `cwd` 会作用到子进程。

In [ ]:
env2 = LocalEnvironment(env={"MY_VAR": "from-config"})
print("env var:", env2.execute({"command": "echo $MY_VAR"})["output"].strip())
print("cwd:", LocalEnvironment(cwd="/tmp").execute({"command": "pwd"})["output"].strip())

## 实验 3：每条命令都是新子 shell

源码用 `subprocess.Popen(..., shell=True)` 每次起一个新进程。所以 `cd` / 环境变量**不跨命令保留**。
亲自验证：两条 execute 之间 `cd` 不生效。

In [ ]:
env3 = LocalEnvironment(cwd=str(REPO))
env3.execute({"command": "cd /tmp"})  # 这个 cd 不会影响下一条
print("下一条仍在:", env3.execute({"command": "pwd"})["output"].strip())

## 实验 4：超时——只是一条 observation，不是 Agent 退出

`timeout=1` 跑 `sleep 2`：命令被杀，返回一条带 `exception_info` 的结果。回忆第 4 章：单命令超时 ≠ Agent 超时。

In [ ]:
out = LocalEnvironment(timeout=1).execute({"command": "printf partial; sleep 2"})
print("returncode:", out["returncode"])
print("exception_info:", out["exception_info"])

## 观察点：为什么杀“进程组”而不是“进程”

看 `_run`：`start_new_session=True` 让命令自成一个进程组；超时用 `os.killpg` 杀**整组**。
为什么？因为命令可能 `sleep 2 & another &` 派生子进程，只杀父进程会留下**孤儿进程**继续占资源。
harness 要保证“一条命令的世界”被干净回收。

In [ ]:
show_source("src/minisweagent/environments/local.py", 72, 92)

## 动手：预测再验证

先猜：`env.execute({"command": "ls /this/does/not/exist"})` 的 returncode 是几？`exception_info` 有内容吗？
写下答案再运行。（提示：命令能启动、只是自己失败——这和“命令根本没跑起来/超时”不同。）

In [ ]:
print(LocalEnvironment().execute({"command": "ls /this/does/not/exist"}))

## 闭卷检查
1. `execute` 的返回 dict 有哪三个关键字段？
2. 为什么 `cd` 不跨命令生效？
3. 超时为什么要 `killpg` 而不是 `kill`？